In [ ]:
!pip install -q transformers

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import (
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    TimesformerForVideoClassification,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMOTIONS = ["happy", "sad", "angry", "fearful", "disgust", "surprised"]
NUM_EMOTIONS = 6

BEST_AUDIO_PATH = "/content/trained_encoders_6emotions/6emo-hubert-er-lr3e5-nf"
BEST_VIDEO_PATH = "/content/trained_encoders_6emotions/6emo-tsf-lr3e5-16f-nf"

print(f"Device: {DEVICE}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    TimesformerForVideoClassification,
)


class SupConCrossModalLoss(nn.Module):
    """Supervised contrastive loss across audio↔video projections (Khosla et al. 2020).

    For anchor i, positives are samples with the same emotion label; negatives are
    everything else in the batch. Symmetric over a→v and v→a directions.

    With τ=0.1 and B=16 random embeddings, loss starts ≈ ln(B) and converges toward
    ln(B/C) at perfect within-class alignment (NOT zero — SupCon's floor on normalized
    embeddings; the relative gap of ln(C) per anchor is what drives learning).
    """

    def __init__(self, weight=1.0, temperature=0.1):
        super().__init__()
        self.weight = weight
        self.tau = temperature

    def forward(self, audio_emb, video_emb, labels):
        a = F.normalize(audio_emb, dim=-1)
        v = F.normalize(video_emb, dim=-1)
        sim_av = a @ v.T / self.tau
        sim_va = v @ a.T / self.tau

        same = (labels[:, None] == labels[None, :]).float()
        pos_count = same.sum(dim=-1)
        valid = pos_count > 1

        log_prob_av = sim_av - torch.logsumexp(sim_av, dim=-1, keepdim=True)
        log_prob_va = sim_va - torch.logsumexp(sim_va, dim=-1, keepdim=True)

        mean_lp_av = (same * log_prob_av).sum(dim=-1) / pos_count.clamp(min=1)
        mean_lp_va = (same * log_prob_va).sum(dim=-1) / pos_count.clamp(min=1)

        if valid.any():
            loss = -0.5 * (mean_lp_av[valid].mean() + mean_lp_va[valid].mean())
        else:
            loss = torch.zeros((), device=audio_emb.device)
        return self.weight * loss


class DifferentiableVideoPreprocess(nn.Module):
    """Replaces HF AutoImageProcessor with differentiable PyTorch ops so
    gradients flow from the emotion loss back through generated frames."""

    def __init__(self, size=224):
        super().__init__()
        self.size = size
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, frames):
        """frames: (B, T, C, H, W) in [0, 1] -> (B, T, C, 224, 224) normalized"""
        B, T, C, H, W = frames.shape
        x = frames.reshape(B * T, C, H, W)
        if H != self.size or W != self.size:
            x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return x.reshape(B, T, C, self.size, self.size)

In [ ]:
def load_frozen_audio_encoder(path, device="cuda"):
    is_hubert = "hubert" in str(path).lower()
    cls = HubertForSequenceClassification if is_hubert else Wav2Vec2ForSequenceClassification
    model = cls.from_pretrained(path)
    model.config.output_hidden_states = True
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    processor = Wav2Vec2FeatureExtractor.from_pretrained(path)
    return model, processor


def load_frozen_video_encoder(path, device="cuda"):
    model = TimesformerForVideoClassification.from_pretrained(path)
    model.config.output_hidden_states = True
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    return model


@torch.no_grad()
def extract_audio_embedding(model, processor, audio_list, sr=16000, window_s=3.0, device="cuda"):
    """Audio embedding is a stop-gradient target for the emotion loss."""
    L = int(window_s * sr)
    wavs = []
    for a in audio_list:
        n = a.numel()
        if n <= L:
            a = F.pad(a, (0, L - n))
        else:
            start = (n - L) // 2
            a = a[start:start + L]
        wavs.append(a.cpu().numpy())
    enc = processor(wavs, sampling_rate=sr, return_tensors="pt",
                    padding=True, truncation=True, max_length=L)
    x = enc["input_values"].to(device)
    mask = enc.get("attention_mask")
    if mask is not None:
        mask = mask.to(device)
    out = model(input_values=x, attention_mask=mask, output_hidden_states=True)
    return out.hidden_states[-1].mean(dim=1)


def extract_video_embedding(model, preprocess, frames, device="cuda"):
    """Differentiable: gradient flows through frames back to the generator."""
    pv = preprocess(frames).to(device)
    out = model(pixel_values=pv, output_hidden_states=True)
    return out.hidden_states[-1].mean(dim=1)

In [ ]:
loss_fn = SupConCrossModalLoss(weight=1.0, temperature=0.1)
labels = torch.tensor([0, 0, 1, 1])
a = torch.randn(4, 768)
v = torch.randn(4, 768)
print(f"SupCon (random, balanced 2 classes): {loss_fn(a, v, labels).item():.4f}")

preprocess = DifferentiableVideoPreprocess(224).to("cpu")
frames = torch.rand(2, 8, 3, 96, 96, requires_grad=True)
out = preprocess(frames)
out.sum().backward()
print(f"Preprocess: {frames.shape} -> {out.shape}, grad flows: {frames.grad is not None}")
print("emotion_utils ready.")

In [ ]:
%%writefile /content/emotion_utils.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    TimesformerForVideoClassification,
)

EMOTIONS = ["happy", "sad", "angry", "fearful", "disgust", "surprised"]
NUM_EMOTIONS = 6


class SupConCrossModalLoss(nn.Module):
    """Supervised contrastive loss across audio↔video projections.

    Positives = same-emotion samples in the batch; negatives = the rest.
    Symmetric over a→v and v→a. With τ=0.1 the loss runs from ≈ ln(B) at
    random down to ≈ ln(B/C) at perfect within-class alignment.
    """

    def __init__(self, weight=1.0, temperature=0.1):
        super().__init__()
        self.weight = weight
        self.tau = temperature

    def forward(self, audio_emb, video_emb, labels):
        a = F.normalize(audio_emb, dim=-1)
        v = F.normalize(video_emb, dim=-1)
        sim_av = a @ v.T / self.tau
        sim_va = v @ a.T / self.tau

        same = (labels[:, None] == labels[None, :]).float()
        pos_count = same.sum(dim=-1)
        valid = pos_count > 1

        log_prob_av = sim_av - torch.logsumexp(sim_av, dim=-1, keepdim=True)
        log_prob_va = sim_va - torch.logsumexp(sim_va, dim=-1, keepdim=True)

        mean_lp_av = (same * log_prob_av).sum(dim=-1) / pos_count.clamp(min=1)
        mean_lp_va = (same * log_prob_va).sum(dim=-1) / pos_count.clamp(min=1)

        if valid.any():
            loss = -0.5 * (mean_lp_av[valid].mean() + mean_lp_va[valid].mean())
        else:
            loss = torch.zeros((), device=audio_emb.device)
        return self.weight * loss


class DifferentiableVideoPreprocess(nn.Module):
    def __init__(self, size=224):
        super().__init__()
        self.size = size
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        x = frames.reshape(B * T, C, H, W)
        if H != self.size or W != self.size:
            x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return x.reshape(B, T, C, self.size, self.size)


def load_frozen_audio_encoder(path, device="cuda"):
    is_hubert = "hubert" in str(path).lower()
    cls = HubertForSequenceClassification if is_hubert else Wav2Vec2ForSequenceClassification
    model = cls.from_pretrained(path)
    model.config.output_hidden_states = True
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    processor = Wav2Vec2FeatureExtractor.from_pretrained(path)
    return model, processor


def load_frozen_video_encoder(path, device="cuda"):
    model = TimesformerForVideoClassification.from_pretrained(path)
    model.config.output_hidden_states = True
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    return model


@torch.no_grad()
def extract_audio_embedding(model, processor, audio_list, sr=16000, window_s=3.0, device="cuda"):
    L = int(window_s * sr)
    wavs = []
    for a in audio_list:
        n = a.numel()
        if n <= L:
            a = F.pad(a, (0, L - n))
        else:
            start = (n - L) // 2
            a = a[start:start + L]
        wavs.append(a.cpu().numpy())
    enc = processor(wavs, sampling_rate=sr, return_tensors="pt",
                    padding=True, truncation=True, max_length=L)
    x = enc["input_values"].to(device)
    mask = enc.get("attention_mask")
    if mask is not None:
        mask = mask.to(device)
    out = model(input_values=x, attention_mask=mask, output_hidden_states=True)
    return out.hidden_states[-1].mean(dim=1)


def extract_video_embedding(model, preprocess, frames, device="cuda"):
    pv = preprocess(frames).to(device)
    out = model(pixel_values=pv, output_hidden_states=True)
    return out.hidden_states[-1].mean(dim=1)